# Graph Traversal

Both traversals visit every vertex and every edge once, so both are **O(V + E)**
on an adjacency list. They differ in the order and in the auxiliary structure:

| | Frontier | Order | Aux space |
|---|---|---|---|
| **BFS** | queue (`deque`) | level by level from the source | O(V) |
| **DFS** | recursion stack (or explicit stack) | as deep as possible first | O(V) |

See [Graph Basics](graph-basics.ipynb) for representations.

![BFS vs DFS Traversal Order](images/bfs-vs-dfs.png)

### Test graph helper

Each notebook is executed standalone by `make test`, so this rebuilds the adjacency
list locally rather than importing it from the basics notebook.

In [ ]:
from collections import deque


def build_adj(n, edges):
    """Adjacency list for an undirected graph on vertices 0..n-1."""
    adj = [[] for _ in range(n)]
    for u, v in edges:
        adj[u].append(v)
        adj[v].append(u)
    return adj


def test_build_adj():
    assert build_adj(3, [(0, 1), (1, 2)]) == [[1], [0, 2], [1]]


test_build_adj()

# Breadth-First Search (BFS)

A queue turns "visit everything" into "visit everything in ring order": dequeue a vertex,
enqueue all its unseen neighbours, repeat. Because the queue is FIFO, every vertex at
distance 1 comes out before any at distance 2 -- which is why BFS finds shortest paths in
unweighted graphs.

Mark a vertex visited **when it is enqueued**, not when it is dequeued. A vertex
reachable from two different neighbours would otherwise be queued twice and emitted
twice.

```
adj: 0:[1,2]  1:[0,2,3]  2:[0,1]  3:[1]

queue [0]      visit 0  → enqueue 1, 2
queue [1,2]    visit 1  → 0 seen, 2 already queued, enqueue 3
queue [2,3]    visit 2  → all seen
queue [3]      visit 3  → all seen

order: 0 1 2 3
```

## Applications
- Find shortest path in unweighted graph
- Web crawlers in search engines
- Peer-to-peer networks
- Social network search
- Garbage collection (Cheney's algorithm)
- Cycle detection
- Ford-Fulkerson algorithm
- Broadcasting in networking

**Time:** O(V + E) &nbsp; **Space:** O(V) for the queue and the visited list

In [ ]:
def bfs(adj, s, visited=None):
    """
    BFS from source vertex s. Returns vertices in visit order.

    Pass an existing `visited` list to continue a traversal across components.

        0
      /   \
    1      2
          / \
         3   4

    From 0: [0, 1, 2, 3, 4]
    """
    if visited is None:
        visited = [False] * len(adj)
    q = deque([s])
    visited[s] = True
    order = []
    while q:
        u = q.popleft()
        order.append(u)
        for v in adj[u]:
            if not visited[v]:
                visited[v] = True  # mark on enqueue, not on dequeue
                q.append(v)
    return order


def test_bfs():
    adj = build_adj(4, [(0, 1), (0, 2), (1, 2), (1, 3)])
    assert bfs(adj, 0) == [0, 1, 2, 3]
    assert bfs(adj, 3) == [3, 1, 0, 2]
    # single vertex, no edges
    assert bfs([[]], 0) == [0]
    # only the source's component is reached
    adj = build_adj(5, [(0, 1), (2, 3)])
    assert bfs(adj, 0) == [0, 1]


test_bfs()

## Disconnected Graphs

One BFS only reaches what is reachable *from its source*. To touch every vertex, loop
over all of them and start a fresh BFS from each one not yet visited.

The `visited` list is threaded through the calls rather than recreated, so no vertex is
processed twice and the total work stays O(V + E) however many components there are.

**Time:** O(V + E) &nbsp; **Space:** O(V)

In [ ]:
def bfs_disconnected(adj):
    """BFS over every component. Returns all vertices in visit order."""
    visited = [False] * len(adj)
    order = []
    for u in range(len(adj)):
        if not visited[u]:
            order += bfs(adj, u, visited)
    return order


def test_bfs_disconnected():
    # two components: {0,1,2,3} and {4,5,6}
    adj = [[1, 2], [0, 3], [0, 3], [1, 2], [5, 6], [4, 6], [4, 5]]
    assert bfs_disconnected(adj) == [0, 1, 2, 3, 4, 5, 6]
    # every vertex appears exactly once
    order = bfs_disconnected([[], [], []])
    assert sorted(order) == [0, 1, 2]


test_bfs_disconnected()

## Counting Connected Components

Exactly the loop above with a counter. The insight: the outer loop can only find an
unvisited vertex when none of the earlier traversals could reach it -- so every time it
does, that is one more component.

**Time:** O(V + E) &nbsp; **Space:** O(V)

In [ ]:
def count_components_bfs(adj):
    """Count connected components in an undirected graph using BFS."""
    visited = [False] * len(adj)
    count = 0
    for u in range(len(adj)):
        if not visited[u]:
            count += 1
            bfs(adj, u, visited)
    return count


def test_count_components_bfs():
    # components: {0,1,2}, {3,4}, {5,6,7}
    adj = [[1, 2], [0, 2], [0, 1], [4], [3], [6, 7], [5], [5]]
    assert count_components_bfs(adj) == 3
    # fully connected
    assert count_components_bfs(build_adj(3, [(0, 1), (1, 2)])) == 1
    # no edges -- every vertex is its own component
    assert count_components_bfs([[], [], []]) == 3
    assert count_components_bfs([]) == 0


test_count_components_bfs()

# Depth-First Search (DFS)

The same "visit everything" job as BFS with the opposite discipline: follow one edge as
deep as it goes, and only back up when stuck. The recursion stack plays the role BFS gives
the queue.

The visit order is the difference that matters, and the diagram at the top of the notebook
contrasts the two: BFS finishes a whole level before descending, while DFS commits to one
branch and runs it to the end.

## Applications
- Cycle detection
- Topological sorting
- Strongly connected components
- Solving maze puzzles
- Path finding

**Time:** O(V + E) &nbsp; **Space:** O(V) -- the recursion can reach depth V on a path
graph

In [ ]:
def dfs_rec(adj, u, visited, order):
    """Recursive DFS helper -- appends vertices to order as they are visited."""
    visited[u] = True
    order.append(u)
    for v in adj[u]:
        if not visited[v]:
            dfs_rec(adj, v, visited, order)


def dfs(adj, s):
    """
    DFS from source vertex s. Returns vertices in visit order.

         0
      /     \
      1      4
      |    /   \
      2   5  -  6
      |
      3

    From 0: [0, 1, 2, 3, 4, 5, 6]
    """
    visited = [False] * len(adj)
    order = []
    dfs_rec(adj, s, visited, order)
    return order


def test_dfs():
    adj = [[1, 4], [0, 2], [1, 3], [2], [0, 5, 6], [4, 6], [4, 5]]
    assert dfs(adj, 0) == [0, 1, 2, 3, 4, 5, 6]
    # goes deep before wide -- contrast with BFS on the same graph
    adj = build_adj(4, [(0, 1), (0, 2), (1, 3)])
    assert dfs(adj, 0) == [0, 1, 3, 2]
    assert bfs(adj, 0) == [0, 1, 2, 3]
    assert dfs([[]], 0) == [0]


test_dfs()

### DFS over components

The same outer loop as BFS: sweep all vertices, start a DFS from each unvisited one.
`count_components_dfs` just adds the counter, and the test asserts both traversals
agree on the count -- the number of components is a property of the graph, not of how
you walk it.

**Time:** O(V + E) &nbsp; **Space:** O(V)

In [ ]:
def dfs_disconnected(adj):
    """DFS over every component. Returns all vertices in visit order."""
    visited = [False] * len(adj)
    order = []
    for u in range(len(adj)):
        if not visited[u]:
            dfs_rec(adj, u, visited, order)
    return order


def count_components_dfs(adj):
    """Count connected components in an undirected graph using DFS."""
    visited = [False] * len(adj)
    count = 0
    for u in range(len(adj)):
        if not visited[u]:
            count += 1
            dfs_rec(adj, u, visited, [])
    return count


def test_dfs_disconnected():
    # components: {0,1,2} and {3,4}
    adj = [[1, 2], [0, 2], [0, 1], [4], [3]]
    assert dfs_disconnected(adj) == [0, 1, 2, 3, 4]
    assert count_components_dfs(adj) == 2
    # BFS and DFS must agree on the number of components
    assert count_components_dfs(adj) == count_components_bfs(adj)
    assert count_components_dfs([[], [], []]) == 3


test_dfs_disconnected()

## Iterative DFS

Python's recursion limit (about 1000 frames) makes recursive DFS unusable on long paths,
so the stack becomes explicit. Two differences from the recursive version:

1. Push neighbours in **reverse** order, since a stack pops last-in-first -- that
   reproduces the recursive visit order
2. A vertex can be pushed more than once before it is popped, so check `visited` again
   *after* popping

The test walks a 2000-vertex path graph, which the recursive version could not handle.

**Time:** O(V + E) &nbsp; **Space:** O(V)

In [ ]:
def dfs_iterative(adj, s):
    """DFS from s using an explicit stack. Returns vertices in visit order."""
    visited = [False] * len(adj)
    stack = [s]
    order = []
    while stack:
        u = stack.pop()
        if visited[u]:  # may have been queued twice before being popped
            continue
        visited[u] = True
        order.append(u)
        # reversed so the first neighbour is popped first, matching dfs_rec
        for v in reversed(adj[u]):
            if not visited[v]:
                stack.append(v)
    return order


def test_dfs_iterative():
    adj = [[1, 4], [0, 2], [1, 3], [2], [0, 5, 6], [4, 6], [4, 5]]
    assert dfs_iterative(adj, 0) == dfs(adj, 0)
    adj = build_adj(4, [(0, 1), (0, 2), (1, 3)])
    assert dfs_iterative(adj, 0) == [0, 1, 3, 2]
    # deep path graph would blow the recursion limit at scale; iterative is fine
    path = build_adj(2000, [(i, i + 1) for i in range(1999)])
    assert dfs_iterative(path, 0) == list(range(2000))


test_dfs_iterative()

# Python Built-in: BFS on a `dict` Graph

There is no graph type in the standard library, but `deque` gives an O(1) `popleft` for
the BFS frontier and `defaultdict(list)` holds the adjacency list. Vertices can be any
hashable value.

The algorithm itself is unchanged apart from two substitutions: a `set` replaces the
boolean visited list (vertices are no longer indices `0..n-1`), and a missing key yields
`[]` instead of raising.

Worth knowing that the `defaultdict` convenience cuts both ways -- reading
`graph[missing]` silently *creates* an empty entry, so the graph can grow just by being
traversed.

**Time:** O(V + E) &nbsp; **Space:** O(V)

In [ ]:
from collections import defaultdict


def bfs_dict(graph, start):
    """BFS over a dict-of-lists graph with arbitrary hashable vertices."""
    visited = {start}
    q = deque([start])
    order = []
    while q:
        node = q.popleft()
        order.append(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                q.append(neighbor)
    return order


def test_bfs_dict():
    graph = defaultdict(list)
    for u, v in [("A", "B"), ("A", "C"), ("B", "D"), ("C", "D")]:
        graph[u].append(v)
        graph[v].append(u)  # undirected
    assert bfs_dict(graph, "A") == ["A", "B", "C", "D"]
    assert bfs_dict(graph, "D") == ["D", "B", "C", "A"]


test_bfs_dict()